[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/signal38/signal38.github.io/blob/main/notebooks/03_evaluate.ipynb)

# Notebook 03 — Three-Model Evaluation

Evaluates three models on the held-out test set (59 weeks):

| Model | Approach |
|-------|----------|
| Naive baseline | Goldstein threshold rule |
| XGBoost | GDELT feature vector |
| LFM2-350M (fine-tuned) | QLoRA knowledge distillation |

**Prerequisites:** Run `01_baseline.ipynb` and `02_finetune.ipynb` first to publish their artifacts.

**Outputs (published to `main`):**
- `data/outputs/results.json` — MAE, accuracy, JSON validity for all three models

In [ ]:
import subprocess, sys, os
from pathlib import Path

REPO = Path('/content/signal38.github.io')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/signal38/signal38.github.io.git', str(REPO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.colab_utils import ensure_notebook_requirements
ensure_notebook_requirements('03_evaluate', requirements_path=str(REPO / 'requirements.txt'))
# Runtime may restart after this cell.

In [ ]:
import subprocess, sys, os, json, re
from pathlib import Path

REPO = Path('/content/signal38.github.io')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/signal38/signal38.github.io.git', str(REPO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.colab_utils import prepare_notebook, require_local_adapter
REPO, PATHS = prepare_notebook(REPO, pull_latest=True)

# Verify prerequisites
require_local_adapter(REPO)  # raises if adapter not published
xgb_model_path = PATHS['xgb_dir'] / 'model.json'
assert xgb_model_path.exists(), f"XGBoost model not found at {xgb_model_path}. Run 01_baseline.ipynb first."

print('Prerequisites satisfied.')

In [ ]:
from scripts.features import cluster_to_features


def load_test_split(split_path, clusters_path):
    """Load test examples joined with their cluster records.

    Returns a list of dicts:
        {'week': str, 'cluster': dict, 'gold_level': int, 'user_content': str}
    """
    clusters = {}
    for line in Path(clusters_path).read_text().splitlines():
        if line.strip():
            c = json.loads(line)
            clusters[c['week_start']] = c

    examples = []
    for line in Path(split_path).read_text().splitlines():
        if not line.strip():
            continue
        ex = json.loads(line)
        user_msg = ex['messages'][1]['content']
        m = re.search(r'Week: (\d{4}-\d{2}-\d{2})', user_msg)
        if not m:
            continue
        week = m.group(1)
        try:
            asst = json.loads(ex['messages'][2]['content'])
            level = int(asst['escalation_level'])
        except (json.JSONDecodeError, KeyError, ValueError):
            continue
        if week not in clusters:
            continue
        examples.append({
            'week': week,
            'cluster': clusters[week],
            'gold_level': level,
            'user_content': user_msg,
        })

    return examples


test_examples = load_test_split(
    PATHS['training_dir'] / 'test.jsonl',
    PATHS['clusters'],
)
gold_labels = [ex['gold_level'] for ex in test_examples]

print(f'Test examples loaded: {len(test_examples)}')

In [ ]:
from scripts.metrics import goldstein_to_escalation, escalation_mae, escalation_accuracy
from scripts.features import cluster_to_features

naive_preds = []
for ex in test_examples:
    cluster = ex['cluster']
    avg_g = cluster.get('stats', {}).get('avg_goldstein', 0.0) or 0.0
    naive_preds.append(goldstein_to_escalation(avg_g))

naive_mae = escalation_mae(naive_preds, gold_labels)
naive_acc = escalation_accuracy(naive_preds, gold_labels)
print(f'Naive baseline — MAE: {naive_mae:.3f}, Accuracy: {naive_acc:.1%}')

In [ ]:
import numpy as np
import xgboost as xgb

xgb_model = xgb.Booster()
xgb_model.load_model(str(xgb_model_path))

X_test = np.array(
    [cluster_to_features(ex['cluster']) for ex in test_examples],
    dtype=np.float32,
)
dtest = xgb.DMatrix(X_test)

# Predictions are 0-indexed; add 1 to restore 1-5 scale
xgb_preds = (xgb_model.predict(dtest).astype(int) + 1).tolist()

xgb_mae = escalation_mae(xgb_preds, gold_labels)
xgb_acc = escalation_accuracy(xgb_preds, gold_labels)
print(f'XGBoost — MAE: {xgb_mae:.3f}, Accuracy: {xgb_acc:.1%}')

In [ ]:
import torch
from tqdm import tqdm
from unsloth import FastLanguageModel
from scripts.metrics import parse_json_output, field_completeness

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(PATHS['adapter_dir']),
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

lfm2_preds = []
lfm2_valid_json = []
lfm2_completeness = []

for ex in tqdm(test_examples, desc='LFM2 inference'):
    messages = [
        {'role': 'system', 'content': 'You are a geopolitical risk analyst specializing in North Korean military activity. Analyze GDELT event data and produce structured risk assessments in JSON format.'},
        {'role': 'user', 'content': ex['user_content']},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    parsed = parse_json_output(generated)

    level = None
    if parsed and 'escalation_level' in parsed:
        try:
            level = int(parsed['escalation_level'])
            if not 1 <= level <= 5:
                level = None
        except (ValueError, TypeError):
            level = None

    lfm2_preds.append(level)
    lfm2_valid_json.append(parsed is not None)
    lfm2_completeness.append(field_completeness(parsed))

    print(f"  week={ex['week']}  gold={ex['gold_level']}  pred={level}  valid_json={parsed is not None}")

lfm2_mae = escalation_mae(lfm2_preds, gold_labels)
lfm2_acc = escalation_accuracy(lfm2_preds, gold_labels)
lfm2_json_rate = sum(lfm2_valid_json) / len(lfm2_valid_json)
lfm2_completeness_mean = sum(lfm2_completeness) / len(lfm2_completeness)

print(f'LFM2-350M — MAE: {lfm2_mae:.3f}, Accuracy: {lfm2_acc:.1%}, '
      f'Valid JSON: {lfm2_json_rate:.1%}, Completeness: {lfm2_completeness_mean:.1%}')

In [ ]:
from scripts.metrics import save_results

print(f'{"Model":<22} {"MAE":>8} {"Accuracy":>10} {"Valid JSON":>12} {"Completeness":>14}')
print('-' * 68)
print(f'{"Naive baseline":<22} {naive_mae:>8.3f} {naive_acc:>10.1%} {"—":>12} {"—":>14}')
print(f'{"XGBoost":<22} {xgb_mae:>8.3f} {xgb_acc:>10.1%} {"—":>12} {"—":>14}')
print(f'{"LFM2-350M":<22} {lfm2_mae:>8.3f} {lfm2_acc:>10.1%} {lfm2_json_rate:>12.1%} {lfm2_completeness_mean:>14.1%}')

results = {
    'test_set_size': len(test_examples),
    'naive_baseline': {
        'mae': naive_mae,
        'accuracy': naive_acc,
    },
    'xgboost': {
        'mae': xgb_mae,
        'accuracy': xgb_acc,
    },
    'lfm2_350m': {
        'mae': lfm2_mae,
        'accuracy': lfm2_acc,
        'valid_json_rate': lfm2_json_rate,
        'field_completeness': lfm2_completeness_mean,
        'parse_failures': int(sum(p is None for p in lfm2_preds)),
    },
}

save_results(results, PATHS['outputs_dir'] / 'results.json')

In [ ]:
from scripts.colab_utils import publish_artifacts

publish_artifacts(
    paths=['data/outputs/results.json'],
    message='Add three-model evaluation results on test set',
    repo_dir=REPO,
)